#### Task 3: Average MonthlyCharges for churned vs non-churned customers (AVERAGEIF equivalent).

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
CANON = ["CustomerID", "Gender", "SeniorCitizen", "Partner", "Dependents", "Tenure", "PhoneService",
         "InternetService", "Contract", "MonthlyCharges", "TotalCharges", "Churn"]
 
 
def load_data():
    """Load the real Telco-Customer-Churn.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("Telco-Customer-Churn.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv", "Telco-Customer-Churn_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {name}: {len(df):,} rows" + ("  (SIMULATED STAND-IN DATA, not the real file)" if "sample" in name else ""))
            break
    else:
        raise FileNotFoundError("Put Telco-Customer-Churn.csv (or run generate_sample_data.py) in the same folder.")
    df = df.rename(columns={c: next((k for k in CANON if k.lower() == c.lower()), c) for c in df.columns})   # customerID -> CustomerID, tenure -> Tenure
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")        # blank strings become NaN
    return df
 
from scipy import stats
 
df = load_data()
 
# Excel equivalent: =AVERAGEIF(Churn_range, "Yes", MonthlyCharges_range)
avg_yes = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].mean()
avg_no = df.loc[df["Churn"] == "No", "MonthlyCharges"].mean()
print(f"\nAverage MonthlyCharges, churned (Yes)     = {avg_yes:.2f}")
print(f"Average MonthlyCharges, not churned (No)  = {avg_no:.2f}")
print(f"Difference (Yes - No)                     = {avg_yes - avg_no:+.2f}  ({(avg_yes / avg_no - 1):+.1%})")
 
# Extra: is the difference statistically significant? (Welch t-test)
t, p = stats.ttest_ind(df.loc[df["Churn"] == "Yes", "MonthlyCharges"], df.loc[df["Churn"] == "No", "MonthlyCharges"], equal_var=False)
print(f"\nWelch t-test: t = {t:.2f}, p = {p:.4g}")
print("Interpretation: customers who churn " + ("pay more" if avg_yes > avg_no else "pay less") + " per month on average"
      + (" and the gap is statistically significant." if p < 0.05 else ", but the gap is not statistically significant."))

Loaded Telco-Customer-Churn_sample.csv: 2,000 rows  (SIMULATED STAND-IN DATA, not the real file)

Average MonthlyCharges, churned (Yes)     = 70.52
Average MonthlyCharges, not churned (No)  = 59.74
Difference (Yes - No)                     = +10.78  (+18.0%)

Welch t-test: t = 7.91, p = 5.673e-15
Interpretation: customers who churn pay more per month on average and the gap is statistically significant.
